In [1]:
import h5py as h5

import os 
import numpy as np 
import nbodykit.lab as NBlab
from astropy.utils.misc import NumpyRNGContext
from nbodykit.hod import Zheng07Model, HODModel
import pickle as pk
from pmesh.pm import ParticleMesh, RealField
from nbodykit.lab import FFTPower, ProjectedFFTPower, ArrayMesh
import Pk_library as PKL
import MAS_library as MASL


In [2]:
import matplotlib.pyplot as pl
%matplotlib inline


In [3]:
# cosmo_params = LH_cosmo_val_all[isim]
fid_cosmo_val_all = np.array([0.3175, 0.049, 0.6711, 0.9624, 0.834])
z = 0.5
# cosmo_params = [0.3175, 0.049, 0.682, 0.6711, 0.96]
Om, Ob, h, ns, s8 = fid_cosmo_val_all
params = {'flat': True, 'H0': 100*h, 'Om0': Om, 'Ob0': Ob, 'sigma8': s8, 'ns': ns}
cosmo_nb = NBlab.cosmology.Planck15.clone(
            h=params['H0']/100., 
            Omega0_b=params['Ob0'], 
            Omega0_cdm=params['Om0']-params['Ob0'],
            n_s=params['ns']) 
Ol = 1 - Om
Hz = 100.0 * np.sqrt(Om * (1. + z)**3 + Ol) # km/s/(Mpc/h)

f = h5.File('/mnt/home/spandey/ceph/CHARM/data/3D_pos_vel_lgM_charm_zsel_0.4_0.7.hdf5', 'r')
box_size = 6000
real_pos = f['pos'][:] + box_size/2
halo_vel = f['vel'][:]
lgmass = f['logM'][:]
f.close()


In [4]:
# np.amin(real_pos-3000), np.amax(real_pos-3000)



In [5]:
group_data = {}  
group_data['Length']    = np.ones(len(real_pos)) * len(real_pos)
group_data['Position']  = real_pos
group_data['Velocity']  = halo_vel
group_data['Mass']      = 10**lgmass
# calculate velocity offset
rsd_factor = (1. + z) / Hz
group_data['VelocityOffset'] = halo_vel * rsd_factor
# save to ArryCatalog for consistency
cat = NBlab.ArrayCatalog(group_data, BoxSize=np.array([box_size, box_size, box_size])) 
cat['Length'] = len(cat)
cat.attrs['rsd_factor'] = rsd_factor 

cat.attrs['Om'] = Om
cat.attrs['Ob'] = Ob
cat.attrs['Ol'] = Ol
cat.attrs['h'] = h 
cat.attrs['ns'] = ns
cat.attrs['s8'] = s8
cat.attrs['Hz'] = Hz # km/s/(Mpc/h)   
halos = NBlab.HaloCatalog(cat, cosmo=cosmo_nb, redshift=0.5, mdef='vir')     
# halos_cats.append(halos)


In [37]:
M_min = 1e12
M1 = 8e14
M0 = 5e13
alpha_sat_fid = 0.5
DlogMmin, Dsigma_logM, DlogM0, DlogM1, Dalpha = 0.0, 0.0, 0.0, 0.0, 0.0
theta_hod = {'logMmin': np.log10(M_min) + DlogMmin, 'sigma_logM': 0.4 + Dsigma_logM, 'logM0': np.log10(M0) + DlogM0, 
                'logM1': np.log10(M1) + DlogM1, 'alpha': alpha_sat_fid + Dalpha}

hod = halos.populate(Zheng07Model, seed=0, **theta_hod)





In [38]:
gal_pos = hod['Position'].compute().astype(np.float32) - box_size/2
gal_vel = hod['Velocity'].compute().astype(np.float32)




In [39]:
# pl.figure()
# _ = pl.hist(gal_vel[:, 0], bins=100, histtype='step', color='k', density=True)
# _ = pl.hist(halo_vel[:, 0], bins=100, histtype='step', color='k', density=True)
np.amin(gal_pos), np.amax(gal_pos), len(gal_pos), len(real_pos)


(-1738.4006, 1738.668, 10682290, 10567003)

In [40]:
chi_gal = np.sqrt(np.sum(gal_pos**2, axis=1))


In [41]:
from colossus.cosmology import cosmology
params = {'flat': True, 'H0': 67.11, 'Om0': 0.3175, 'Ob0': 0.049, 'sigma8': 0.834, 'ns': 0.9624}
cosmo = cosmology.setCosmology('myCosmo', **params)
import scipy as sp
z_array = np.linspace(0.3, 0.8, 200)
chi_array = cosmo.comovingDistance(0.0, z_array)
z_interp = sp.interpolate.interp1d(np.log(chi_array), z_array)
z_gal = z_interp(np.log(chi_gal))



In [42]:
import os, sys
import numpy as np
import healpy as hp
import numexpr as ne 

def rd2tp(ra,dec):
    """Convert ra,dec -> tht,phi"""
    tht = (-dec+90.0)/180.0*np.pi
    phi = ra/180.0*np.pi
    return tht,phi

def tp2rd(tht,phi):
    """Convert tht,phi -> ra,dec"""
    ra  = phi/np.pi*180.0
    dec = -1*(tht/np.pi*180.0-90.0)
    return ra,dec

ux   = gal_pos[:,0]/chi_gal
uy   = gal_pos[:,1]/chi_gal
uz   = gal_pos[:,2]/chi_gal
vec_pos = np.c_[ux,uy,uz]
tht,phi = hp.vec2ang(vec_pos)
ra_gal,dec_gal  = tp2rd(tht,phi)



In [43]:
v_los_gal = gal_vel[:,0]*ux + gal_vel[:,1]*uy + gal_vel[:,2]*uz



In [44]:
f = h5.File('/mnt/home/spandey/ceph/CHARM/data/pos_ra_dec_z_galaxies_zsel_0.4_0.7.hdf5', 'w')
# f.create_dataset('pos', data=pos_h_mock_sel)
f.create_dataset('ra', data=ra_gal)
f.create_dataset('dec', data=dec_gal)
f.create_dataset('z', data=z_gal)
vel = f.create_dataset('v_los', data=v_los_gal)
vel.attrs['description'] = 'velocity along the line of sight of halos in km/s'
# lg10M = f.create_dataset('logM', data=lgMass_mock_sel)
# lg10M.attrs['description'] = 'log10 of the halo masses in Msun/h. Minimum halo mass is 10^12.7 Msun/h'
f.close()
